# 04 â€” Feature Engineering

Two questions: (1) do derived features (loan-to-income, payment-to-income, credit history length) help? (2) do previously-dropped raw features (`mort_acc`, `pub_rec_bankruptcies`, `application_type`) help? Hold the model fixed (LightGBM defaults) and vary the feature set.

## Setup

In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

FEATURES = [
    'loan_amnt', 'term', 'int_rate', 'installment', 'grade',
    'emp_length', 'home_ownership', 'annual_inc', 'verification_status',
    'purpose', 'addr_state', 'dti', 'delinq_2yrs', 'fico_range_low',
    'inq_last_6mths', 'open_acc', 'pub_rec', 'revol_bal', 'revol_util',
    'total_acc',
]
CATEGORICAL = ['term', 'grade', 'home_ownership', 'verification_status', 'purpose', 'addr_state']

def parse_term(s):
    if isinstance(s, str):
        return int(s.strip().split()[0])
    return np.nan

def parse_emp_length(s):
    if not isinstance(s, str):
        return np.nan
    s = s.strip()
    if '<' in s:
        return 0
    if '+' in s:
        return 10
    parts = s.split()
    return int(parts[0]) if parts and parts[0].isdigit() else np.nan

def parse_pct(s):
    if isinstance(s, str):
        s = s.replace('%', '').strip()
        return float(s) if s else np.nan
    return s

# Load and clean
df = pd.read_csv('../data/loan.csv', low_memory=False)
df = df[df['loan_status'].isin(['Fully Paid', 'Charged Off'])].copy()
df['target'] = (df['loan_status'] == 'Charged Off').astype(int)
df, _ = train_test_split(df, train_size=0.10, stratify=df['target'], random_state=42)

df['term'] = df['term'].map(parse_term)
df['int_rate'] = df['int_rate'].map(parse_pct)
df['revol_util'] = df['revol_util'].map(parse_pct)
df['emp_length'] = df['emp_length'].map(parse_emp_length)

# Data-quality fixes (per notebook 01 findings)
df.loc[df['dti'] > 50, 'dti'] = np.nan
df.loc[df['revol_util'] > 100, 'revol_util'] = np.nan

X = df[FEATURES].copy()
for col in CATEGORICAL:
    X[col] = X[col].astype('category')
y = df['target']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print(f'Train: {len(X_train):,}  Test: {len(X_test):,}  Default rate: {y_train.mean():.1%}')


Train: 107,624  Test: 26,907  Default rate: 20.0%


## Add derived features and re-run

In [2]:
import lightgbm as lgb
from sklearn.metrics import roc_auc_score
import warnings
warnings.filterwarnings('ignore')

# Re-load with extra columns this time
df_full = pd.read_csv('../data/loan.csv', low_memory=False)
df_full = df_full[df_full['loan_status'].isin(['Fully Paid', 'Charged Off'])].copy()
df_full['target'] = (df_full['loan_status'] == 'Charged Off').astype(int)
df_full, _ = train_test_split(df_full, train_size=0.10, stratify=df_full['target'], random_state=42)

# Same parsing as before
df_full['term'] = df_full['term'].map(parse_term)
df_full['int_rate'] = df_full['int_rate'].map(parse_pct)
df_full['revol_util'] = df_full['revol_util'].map(parse_pct)
df_full['emp_length'] = df_full['emp_length'].map(parse_emp_length)
df_full.loc[df_full['dti'] > 50, 'dti'] = np.nan
df_full.loc[df_full['revol_util'] > 100, 'revol_util'] = np.nan

# Derived features
df_full['loan_to_income'] = df_full['loan_amnt'] / df_full['annual_inc'].replace(0, np.nan)
df_full['payment_to_income'] = (df_full['installment'] * 12) / df_full['annual_inc'].replace(0, np.nan)
df_full['credit_history_years'] = (
    pd.to_datetime(df_full['issue_d'], format='%b-%Y', errors='coerce')
    - pd.to_datetime(df_full['earliest_cr_line'], format='%b-%Y', errors='coerce')
).dt.days / 365.25

DERIVED = ['loan_to_income', 'payment_to_income', 'credit_history_years']
DROPPED_RAW_NUMERIC = ['mort_acc', 'pub_rec_bankruptcies']
DROPPED_RAW_CAT = ['application_type']

# Confirm those columns exist and have data
for c in DROPPED_RAW_NUMERIC + DROPPED_RAW_CAT:
    if c in df_full.columns:
        print(f'{c}: {df_full[c].isna().sum() / len(df_full):.1%} missing')

# Cast new categoricals
for col in CATEGORICAL + DROPPED_RAW_CAT:
    if col in df_full.columns:
        df_full[col] = df_full[col].astype('category')

def evaluate_features(features, cat_cols, name):
    X = df_full[features].copy()
    for c in cat_cols:
        if c in X.columns:
            X[c] = X[c].astype('category')
    y = df_full['target']
    X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
    model = lgb.LGBMClassifier(n_estimators=500, learning_rate=0.05, num_leaves=63,
                                min_child_samples=50, random_state=42, n_jobs=-1, verbose=-1)
    model.fit(X_tr, y_tr, eval_set=[(X_te, y_te)],
              callbacks=[lgb.early_stopping(30), lgb.log_evaluation(0)])
    proba = model.predict_proba(X_te)[:, 1]
    return {'feature_set': name, 'n_features': len(features), 'auc': roc_auc_score(y_te, proba)}

results = []
results.append(evaluate_features(FEATURES, CATEGORICAL, 'baseline_20'))
print(' baseline done')
results.append(evaluate_features(FEATURES + DERIVED, CATEGORICAL, '+derived'))
print(' +derived done')
extra = [c for c in DROPPED_RAW_NUMERIC + DROPPED_RAW_CAT if c in df_full.columns]
results.append(evaluate_features(FEATURES + extra, CATEGORICAL + DROPPED_RAW_CAT, '+dropped_raw'))
print(' +dropped done')
results.append(evaluate_features(FEATURES + DERIVED + extra, CATEGORICAL + DROPPED_RAW_CAT, 'all'))
print(' all done')

results_df = pd.DataFrame(results)
results_df['auc'] = results_df['auc'].round(4)
print('\nResults:')
print(results_df.to_string(index=False))


mort_acc: 3.6% missing
pub_rec_bankruptcies: 0.0% missing
application_type: 0.0% missing


Training until validation scores don't improve for 30 rounds


Early stopping, best iteration is:
[131]	valid_0's binary_logloss: 0.452473
 baseline done


Training until validation scores don't improve for 30 rounds


Early stopping, best iteration is:
[135]	valid_0's binary_logloss: 0.451917
 +derived done


Training until validation scores don't improve for 30 rounds


Early stopping, best iteration is:
[134]	valid_0's binary_logloss: 0.451241
 +dropped done


Training until validation scores don't improve for 30 rounds


Early stopping, best iteration is:
[130]	valid_0's binary_logloss: 0.450691
 all done

Results:
 feature_set  n_features    auc
 baseline_20          20 0.7132
    +derived          23 0.7148
+dropped_raw          23 0.7157
         all          26 0.7168


**Observations**

| Feature set | n_features | AUC |
|---|---|---|
| baseline_20 (current) | 20 | 0.7132 |
| +derived (loan_to_income, payment_to_income, credit_history_years) | 23 | 0.7148 |
| +dropped_raw (mort_acc, pub_rec_bankruptcies, application_type) | 23 | 0.7157 |
| all | 26 | 0.7168 |

- Both feature additions help, modestly and additively. Combined: **+0.0036 AUC** over the current 20-feature baseline.
- The previously-dropped raw features (`mort_acc`, `pub_rec_bankruptcies`, `application_type`) help slightly more than the derived ones. Worth noting — the "obvious" derived features (loan-to-income ratios) didn't outperform just adding more raw signals.
- `mort_acc` (mortgage account count) is the most useful single addition; intuitively, owning real estate is a strong financial-stability signal that wasn't fully captured by `home_ownership='MORTGAGE'`.


## Bottom line

**Adding all 6 features gets AUC 0.7168, +0.0036 over the current 20-feature model.**

That's a small but real improvement. Worth shipping if I were doing a v2 of train.py — the cost is just adding 6 lines to the FEATURES list. Whether to do it depends on whether 0.0036 AUC is worth the disruption to a deployed model. For this portfolio piece, I'd note the finding in the methodology and leave the deployed v1 alone.
